# GAN Y Transformer

# Con GAN (Generative Adversarial Network)

In [1]:
import torch
import torch.nn as nn

import torch.nn.init as init

class Generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Generator, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 256)
        self.fc3 = nn.Linear(256, 512)
        self.fc4 = nn.Linear(512, output_dim)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

        # Inicialización de los pesos
        self._initialize_weights()

    def _initialize_weights(self):
        # Inicialización de los pesos de las capas usando una distribución normal
        for layer in self.children():
            if isinstance(layer, nn.Linear):
                init.normal_(layer.weight, mean=0.0, std=0.02)  # Inicialización normal
                if layer.bias is not None:
                    init.constant_(layer.bias, 0)

    def forward(self, z):
        x = self.relu(self.fc1(z))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.tanh(self.fc4(x))  # Salida entre -1 y 1
        return x


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)  # No aplicar sigmoid aquí aún
        return self.sigmoid(x)  # Aplicar Sigmoid justo antes de la salida

# Dimensiones
latent_dim = 200 
input_dim = df_int_sin_date.shape[1]  # Número de características en los datos

# Crear el generador y el discriminador
generator = Generator(latent_dim, input_dim)
discriminator = Discriminator(input_dim)

# Definir las funciones de pérdida y optimizadores
adversarial_loss = nn.BCELoss()  # Pérdida binaria para GAN
optimizer_g = torch.optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.999))  # Tasa de aprendizaje más baja
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))  # Tasa de aprendizaje más baja

epochs = 100
batch_size = 20

# Convertir los datos escalados a un tensor de PyTorch
df_tensor = torch.tensor(df_int_sin_date.values, dtype=torch.float32)

for epoch in range(epochs):
    for i in range(0, len(df_tensor), batch_size):
        real_data = df_tensor[i:i+batch_size]
        batch_size = real_data.size(0)

        # Etiquetas reales y falsas
        real_labels = torch.ones(batch_size, 1)
        fake_labels = torch.zeros(batch_size, 1)

        # ===== Entrenar el discriminador =====
        optimizer_d.zero_grad()

        # Pérdida para datos reales
        real_preds = discriminator(real_data)
        real_loss = adversarial_loss(real_preds, real_labels)

        # Pérdida para datos generados
        z = torch.randn(batch_size, latent_dim)
        fake_data = generator(z)
        fake_preds = discriminator(fake_data.detach())  # No calcular gradientes para los datos generados
        fake_loss = adversarial_loss(fake_preds, fake_labels)

        # Pérdida total para el discriminador
        d_loss = real_loss + fake_loss
        d_loss.backward()
        optimizer_d.step()

        # ===== Entrenar el generador =====
        optimizer_g.zero_grad()

        # Intentar que el generador engañe al discriminador
        fake_preds = discriminator(fake_data)
        g_loss = adversarial_loss(fake_preds, real_labels)  # Queremos que el discriminador piense que los datos generados son reales

        g_loss.backward()
        optimizer_g.step()

    if epoch % 20 == 0:
        print(f"Epoch [{epoch}/{epochs}] | D Loss: {d_loss.item()} | G Loss: {g_loss.item()}")

# Generar nuevos datos
z = torch.randn(200, latent_dim)  # 200 nuevas filas
generated_data = generator(z).detach().numpy()

# Convertir a DataFrame
generated_df = pd.DataFrame(generated_data, columns=df_int_sin_date.columns)


NameError: name 'df_int_sin_date' is not defined

# Merge

In [ ]:

# Mostrar los nuevos datos generados
generated_df = generated_df.clip(lower=0)
generated_df

In [2]:
generated_data = generated_data.clip(lower=0)
generated_data

NameError: name 'generated_data' is not defined

In [ ]:
import pandas as pd

start_date = pd.to_datetime("2023-06-27 01:00:00+00:00")

new_dates = pd.date_range(start=start_date, periods=200, freq='H')

generated_data['date'] = new_dates

generated_data.insert(0, 'date', generated_data.pop('date'))

df_int_aug = pd.concat([df_int, generated_data], ignore_index=True)
df_int_aug

In [3]:
set(df_int["date"]).issubset(set(df_orig["date"]))

NameError: name 'df_int' is not defined

In [ ]:
df_orig[df_orig["date"] == df_int_aug["date"][0]]

In [ ]:
df_orig[df_orig["date"] == df_int_aug["date"][980]]

In [ ]:
df_orig[df_orig["date"] == df_int_aug["date"][1534]]

In [ ]:
df = pd.merge(df_orig, df_int_aug, how="left", on=["date"])

In [ ]:
nuevas_columnas = df_int_aug.columns.difference(df_orig.columns)

In [ ]:
nuevas_columnas

In [ ]:
df.loc[5798:5850, ["date"] + list(nuevas_columnas) + columns]

In [ ]:
sum(df["eBC_tot"].notna())

## Matriz de correlación entre vehículos y variables de contaminación

In [ ]:
import seaborn as sns

# Calcular la matriz de correlación
cols = [col for col in periodo_1_imputed.columns if col != 'date']
correlation_matrix = periodo_1_imputed[cols].corr()

# Mostrar la matriz
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlación entre Variables de Contaminación y Meteorológicas')
plt.show()